In [2]:
import pyspark
import os

In [3]:
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("Spark-Developer").getOrCreate()
# # spark = SparkSession.builder.appName("Spark-Developer").master("local[*]").getOrCreate()
# spark

In [4]:
from app.utils.spark_session import spark, display_df
from pyspark.sql import functions as F
from pyspark.sql import Window

spark

In [7]:
%load_ext sparksql_magic

In [12]:
%%sparksql
show tables in my_database;

namespace,tableName,isTemporary
my_database,sales_enriched,False
my_database,user_events,False
my_database,orders_iceberg,False


In [29]:
%%sparksql
with most_status as (
    select
        quantity, status, count(1) as cnt
    from my_database.orders_iceberg
    group by 1,2
),
ranked_status as (
    select *, row_number() over(partition by quantity order by cnt desc) as rn
    from most_status
),
stats as (
    select quantity, count(status) as tot_cnt, count(distinct status) as uniq_cnt
    from my_database.orders_iceberg
    group by 1
)

select s.* , rs.status as most_status
from stats s
join ranked_status rs
on s.quantity = rs.quantity
where rs.rn = 1

quantity,tot_cnt,uniq_cnt,most_status
1,6,2,completed
2,3,3,completed
3,1,1,shipped


In [5]:
df = spark.read.table('my_database.orders_iceberg')
df

order_id,customer_name,product,quantity,unit_price,order_date,status
1,Alice Johnson,Laptop,1,1299.99,2024-01-15,completed
2,Bob Smith,Mouse,2,29.99,2024-01-16,completed
3,Carol White,Keyboard,1,79.99,2024-01-17,pending
4,David Lee,Monitor,1,399.99,2024-01-18,completed
5,Eve Martinez,Headphones,3,149.99,2024-01-19,shipped
6,Frank Brown,Webcam,2,89.99,2024-01-20,pending
7,Grace Kim,USB Hub,1,49.99,2024-01-21,completed
8,Henry Wilson,Laptop Stand,2,59.99,2024-01-22,shipped
9,Iris Chen,SSD Drive,1,119.99,2024-01-23,completed
10,James Davis,Microphone,1,199.99,2024-01-24,pending


In [71]:
stats = df.groupBy('quantity').agg(F.count(df.status).alias("tot_cnt"), F.count_distinct(df.status).alias("uniq_cnt"))

window = Window.partitionBy('quantity').orderBy(F.col('cnt').desc())

most_status = df.groupBy('quantity', 'status') \
    .agg(count(F.lit(1)).alias('cnt')) \
    .withColumn('rn', F.row_number().over(window)) \
    .filter('rn = 1').selectExpr('quantity', 'status as most_status')

_df = stats.join(most_status, on='quantity', how='inner')
_df

quantity,tot_cnt,uniq_cnt,most_status
1,6,2,completed
3,1,1,shipped
2,3,3,completed


In [11]:
df = spark.read.csv('./app/data/source/customers.csv', header=True)
df.show(10)

+-----------+-------------+--------------------+--------+-------------+-----+---------------+----------+
|customer_id|         name|               email|   phone|         city|state|membership_tier| join_date|
+-----------+-------------+--------------------+--------+-------------+-----+---------------+----------+
|       C001|Alice Johnson|alice.johnson@exa...|555-0101|San Francisco|   CA|           Gold|2022-01-15|
|       C002|    Bob Smith|bob.smith@example...|555-0102|     New York|   NY|         Silver|2021-06-20|
|       C003|  Carol White|carol.white@examp...|555-0103|      Chicago|   IL|         Bronze|2023-03-10|
|       C004|    David Lee|david.lee@example...|555-0104|       Austin|   TX|           Gold|2020-11-05|
|       C005| Eve Martinez|eve.martinez@exam...|555-0105|      Seattle|   WA|       Platinum|2019-08-22|
|       C006|  Frank Brown|frank.brown@examp...|555-0106|       Boston|   MA|         Silver|2022-07-18|
|       C007|    Grace Kim|grace.kim@example...|555-010

In [12]:
df.createOrReplaceTempView('customers_df')

In [20]:
sql = """
SELECT *
FROM customers_df
QUALIFY ROW_NUMBER() OVER(PARTITION BY membership_tier ORDER BY join_date) = 1
"""
spark.sql(sql)

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near 'ROW_NUMBER'. SQLSTATE: 42601 (line 4, pos 8)

== SQL ==

SELECT *
FROM customers_df
QUALIFY ROW_NUMBER() OVER(PARTITION BY membership_tier ORDER BY join_date) = 1
--------^^^


JVM stacktrace:
org.apache.spark.sql.catalyst.parser.ParseException
	at org.apache.spark.sql.catalyst.parser.ParseException.withCommand(parsers.scala:285)
	at org.apache.spark.sql.catalyst.parser.AbstractParser.parse(parsers.scala:97)
	at org.apache.spark.sql.execution.SparkSqlParser.parse(SparkSqlParser.scala:54)
	at org.apache.spark.sql.catalyst.parser.AbstractSqlParser.parsePlan(AbstractSqlParser.scala:93)
	at org.apache.spark.sql.catalyst.parser.extensions.IcebergSparkSqlExtensionsParser.parsePlan(IcebergSparkSqlExtensionsParser.scala:122)
	at io.delta.sql.parser.DeltaSqlParser.$anonfun$parsePlan$1(DeltaSqlParser.scala:86)
	at io.delta.sql.parser.DeltaSqlParser.parse(DeltaSqlParser.scala:119)
	at io.delta.sql.parser.DeltaSqlParser.parsePlan(Delta

In [17]:
df.agg(F.count_distinct('state'), F.count('state'))

count(DISTINCT state),count(state)
16,20
